<a href="https://colab.research.google.com/github/tracykeeya/nlp_project_group_Sandra_Tracy/blob/main/Group_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets huggingface_hub gradio fastapi uvicorn pypdf sentence-transformers

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")      # read from Colab Secrets, never paste tokens in code
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: Gumosh


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

LOCAL_MODEL_DIR = "/content/drive/MyDrive/model_bert"   # TODO: path to your saved model

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(clf("Terrible experience, I want my money back."))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9973537921905518}]


In [ ]:
MODEL_NAME = "model_bert"    # TODO: change to your model
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

model.push_to_hub(MODEL_ID)
tokenizer.push_to_hub(MODEL_ID)
print("Model page:", f"https://huggingface.co/{MODEL_ID}")
#pushing the model to hugging face

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lrlfadc/model.safetensors:   3%|2         | 7.99MB /  268MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model page: https://huggingface.co/Gumosh/model_bert


In [ ]:
#Write a model card
#A model card tells people what your model does and where it fails. Fill in your real numbers.

from huggingface_hub import ModelCard

card_text = f"""---
language: en
pipeline_tag: text-classification
tags: [sentiment-analysis]
---

# {MODEL_NAME}

Sentiment classifier (positive / negative) fine-tuned from `distilbert-base-uncased`.

**Group members:** Gumoshabe Sandra and Keeya Tracy
**Dataset:** stanfordnlp/imdb

## Results on the test set
| Metric | Score |
|---|---|
| Accuracy | 0.9090 |
| F1 | 0.9090 |

## Example predictions
1. "Terrible experience, I want my money back." -> Negative
2. "This is boring" -> Negative
3. "Absolutely wonderful movie with great acting!" -> Positive

## Limitations
Trained on English movie reviews only; struggles with sarcasm and highly complex or mixed language text.
"""

card = ModelCard(card_text)
card.push_to_hub(MODEL_ID)
print("Model card uploaded.")

No files have been modified since last commit. Skipping to prevent empty commit.


Model card uploaded.


In [ ]:
hub_clf = pipeline("text-classification", model=MODEL_ID)
hub_clf("I really enjoyed this class!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.9982253909111023}]

In [ ]:
 #Build a Gradio demo
#Gradio turns a Python function into a web app. First test it here in Colab.
import gradio as gr

def predict(text):
    if not text.strip():
        return {}
    result = hub_clf(text)[0]
    score = result["score"]
    # Gradio's Label component shows a confidence bar for each class
    other = "NEGATIVE" if result["label"] == "POSITIVE" else "POSITIVE"
    return {result["label"]: score, other: 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    description="Type a sentence and see if it is positive or negative.",
    examples=["The food was amazing!", "Worst service ever.", "It was okay, nothing special."],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1b3278f5e472dfa47e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile app.py
import gradio as gr
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id
clf = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {}
    r = clf(text)[0]
    other = "NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"
    return {r["label"]: r["score"], other: 1 - r["score"]}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    examples=["The food was amazing!", "Worst service ever."],
)
demo.launch()

Overwriting app.py


In [62]:
%%writefile requirements.txt
transformers
torch

Writing requirements.txt


In [ ]:
#from huggingface_hub import HfApi
#api = HfApi()

# Replace the placeholder model id inside app.py with yours
#app_code = open("app.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID)
#open("app.py", "w").write(app_code)

#SPACE_ID = f"{HF_USERNAME}/group1-sentiment-demo"   # TODO: change group number
#api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
#api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
#api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=SPACE_ID, repo_type="space")
#print("Your Space (takes 2-5 minutes to build):", f"https://huggingface.co/spaces/{SPACE_ID}")

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id

app = FastAPI(title="Sentiment API")

# Load the model ONCE when the app starts, not on every request (that would be very slow)
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))

Overwriting main.py


In [63]:
# Put your model id into main.py, then start the API in the background
open("main.py", "w").write(open("main.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID))

import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)   # give it time to load the model
print("API started")

API started


In [64]:
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

for sentence in ["I love this app!", "This is so frustrating.", ""]:
    r = requests.post(f"{BASE}/predict", json={"text": sentence})
    print(r.status_code, r.json())

{'status': 'ok'}
200 {'label': 'LABEL_1', 'score': 0.9983}
200 {'label': 'LABEL_0', 'score': 0.998}
400 {'detail': 'Text cannot be empty'}


In [65]:
server.terminate()   # stop the API when you are done

**PART B:CHATBOT**


In [68]:
from transformers import pipeline

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # change if unavailable
llm = pipeline("text-generation", model=LLM_MODEL, torch_dtype="auto", device_map="auto")

def ask_llm(messages, max_tokens=400):
    response = llm(messages, max_new_tokens=max_tokens, do_sample=True, temperature=0.2)
    return response[0]["generated_text"][-1]["content"]

print(ask_llm([
    {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
    {"role": "user", "content": "What is natural language processing?"},
]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Natural Language Processing (NLP) is the branch of artificial intelligence that enables computers to understand, interpret, and generate human language.


In [69]:
# Compare: your fine-tuned model vs the LLM
#The LLM can do sentiment analysis with no training at all, just a prompt. But is it better, faster, or cheaper?

import time, json
from transformers import pipeline

# Fallback definition if hub_clf is not yet loaded from previous steps
if 'hub_clf' not in globals():
    print("hub_clf was not found. Loading a default fine-tuned sentiment classifier ('distilbert-base-uncased-finetuned-sst-2-english')...")
    hub_clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

test_sentences = [
    "The delivery was quick and the packaging was great.",
    "I waited two hours and nobody helped me.",
    "Not bad, but I expected more for the price.",
    "Absolutely brilliant lecture today!",
    "The app keeps crashing every time I open it.",
]

for s in test_sentences:
    t0 = time.time(); mine = hub_clf(s)[0]["label"]; t_mine = time.time() - t0
    t0 = time.time()
    llm_label = ask_llm([
        {"role": "system", "content": "Classify the sentiment. Reply with ONE word: POSITIVE or NEGATIVE."},
        {"role": "user", "content": s},
    ], max_tokens=5).strip()
    t_llm = time.time() - t0
    print(f"{s[:45]:<47} mine={mine:<9} ({t_mine:.2f}s)  llm={llm_label:<9} ({t_llm:.2f}s)")

The delivery was quick and the packaging was    mine=LABEL_1   (0.01s)  llm=POSITIVE  (0.23s)
I waited two hours and nobody helped me.        mine=LABEL_0   (0.01s)  llm=NEGATIVE  (0.20s)
Not bad, but I expected more for the price.     mine=LABEL_0   (0.01s)  llm=NEGATIVE  (0.22s)
Absolutely brilliant lecture today!             mine=LABEL_1   (0.01s)  llm=POSITIVE  (0.20s)
The app keeps crashing every time I open it.    mine=LABEL_1   (0.01s)  llm=NEGATIVE  (0.21s)
